# DeepChem Introduction & Setup: Point-by-Point Code

This notebook demonstrates the fundamental concepts outlined in the DeepChem Introduction syllabus. We will go through Dataset Handling, Molecular Featurization, Model Building, Training, and Evaluation.

## 1. Setup and Imports
Importing `deepchem` and `rdkit` which are the core components.

In [ ]:
import deepchem as dc
from rdkit import Chem
import numpy as np

print("DeepChem version:", dc.__version__)

## 2. Molecular Featurization (RDKit Integration)
DeepChem uses `Featurizers` to convert SMILES strings or RDKit molecules into ML-compatible numerical representations like Fingerprints.

In [ ]:
smiles = ['C1=CC=CC=C1', 'O=C(C)Oc1ccccc1C(=O)O'] # Benzene, Aspirin

# 2D RDKit Object
mols = [Chem.MolFromSmiles(s) for s in smiles]

# DeepChem Featurizer: Circular Fingerprint (Morgan Fingerprint)
featurizer = dc.feat.CircularFingerprint(size=1024)
features = featurizer.featurize(mols)

print("Shape of extracted features:", features.shape)
print("Features for Benzene:", features[0][:50], "...")

## 3. Dataset Handling (MoleculeNet)
DeepChem provides `MoleculeNet` to easily load chemical datasets. We will load the **Tox21** dataset (Toxicity Prediction) or **Delaney** dataset (Solubility Prediction).

In [ ]:
# Load Delaney solubility dataset
# This automatically featurizes the dataset and splits it into train/valid/test sets
tasks, datasets, transformers = dc.molnet.load_delaney(featurizer='GraphConv')
train_dataset, valid_dataset, test_dataset = datasets

print("Tasks:", tasks)
print("Train dataset shape:", train_dataset.X.shape)
print("Test dataset shape:", test_dataset.X.shape)

## 4. Building the Model (Graph Neural Network)
DeepChem makes it easy to build powerful Deep Learning architectures like **Graph Convolutional Networks (GNN)**.

In [ ]:
# Create a Graph Convolutional Model for Regression (predicting solubility)
model = dc.models.GraphConvModel(n_tasks=len(tasks), mode='regression', dropout=0.2)

## 5. Training & Evaluation
Train the model and measure its performance using standard metrics (like Pearson R2 or RMSE).

In [ ]:
# Train the model (fit)
print("Training model...")
model.fit(train_dataset, nb_epoch=50)

# Define evaluation metric
metric = dc.metrics.Metric(dc.metrics.pearson_r2_score)

# Evaluate
train_scores = model.evaluate(train_dataset, [metric], transformers)
test_scores = model.evaluate(test_dataset, [metric], transformers)

print("Train Score:", train_scores)
print("Test Score:", test_scores)

## 6. Prediction (Virtual Screening / Property Prediction)
Use the trained AI model to predict properties for new, unseen molecules.

In [ ]:
# Define new molecules for prediction
new_smiles = ['CC(=O)OC1=CC=CC=C1C(=O)O', 'CCO'] # Aspirin and Ethanol

# Featurize new molecules
featurizer = dc.feat.ConvMolFeaturizer()
new_mols = [Chem.MolFromSmiles(s) for s in new_smiles]
new_features = featurizer.featurize(new_mols)

# Create a DeepChem NumpyDataset
new_dataset = dc.data.NumpyDataset(X=new_features)

# Predict using our trained GNN
predictions = model.predict(new_dataset)

# Un-transform the predictions to get original units
for transformer in transformers:
    predictions = transformer.untransform(predictions)

for s, p in zip(new_smiles, predictions):
    print(f"Predicted Solubility for {s}: {p[0]}")